# Hello VLA

Welcome to your very first hands on lesson. In this notebook you will:

1. Confirm your Colab runtime is working.
2. Load a real multimodal model (SmolVLM) and ask it to describe an image.
3. Try to load a real Vision-Language-Action model (SmolVLA) and inspect it.

You do not need to understand the code yet. Run each cell in order by pressing **Shift + Enter**.

## Step 1. Turn on a free GPU (optional but faster)

In the Colab menu, click **Runtime** then **Change runtime type**. Under **Hardware accelerator**, pick **T4 GPU**, then click **Save**.

If Colab says no GPU is available right now, that is fine. Everything in this notebook will still run on CPU, just slower.

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
print('PyTorch version:', torch.__version__)

## Step 2. Load SmolVLM, the vision language brain of SmolVLA

A VLA is made of two halves: a **vision language model** that understands images and text, and an **action head** that turns that understanding into robot actions.

We are going to start by loading the vision language half on its own. It is called **SmolVLM**. We will give it an image and a question, and it will answer in plain English.

In [ ]:
!pip install -q -U transformers

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = 'HuggingFaceTB/SmolVLM-500M-Instruct'
dtype = torch.bfloat16 if device == 'cuda' else torch.float32

processor = AutoProcessor.from_pretrained(model_id)
vlm = AutoModelForImageTextToText.from_pretrained(model_id, torch_dtype=dtype).to(device)

n_params = sum(p.numel() for p in vlm.parameters())
print(f'Loaded {model_id}')
print(f'Parameters: {n_params/1e6:.1f}M')

In [ ]:
from PIL import Image
import requests

image_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/640px-Cat03.jpg'
headers = {'User-Agent': 'vla-course/1.0'}
image = Image.open(requests.get(image_url, headers=headers, stream=True).raw).convert('RGB')
image.thumbnail((512, 512))
image

In [ ]:
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': 'Describe this scene in one sentence, then list three objects you can see.'}
        ]
    }
]

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors='pt').to(device)

with torch.no_grad():
    generated = vlm.generate(**inputs, max_new_tokens=120)

new_tokens = generated[:, inputs['input_ids'].shape[1]:]
answer = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
print(answer.strip())

Stop and notice what just happened. You gave a neural network an image plus a question in English, and it answered in English. That was not possible five years ago. This exact model, or a close cousin, is the brain of every modern VLA.

Imagine the image above is the view from a robot's camera sitting above a table. The same model could describe the scene, identify objects, and answer questions about them. The only piece missing is a small network that turns that understanding into joint angles. That piece is called the **action expert**, and together they form a VLA.

## Step 3. Peek inside a real VLA (optional)

The cell below tries to install the LeRobot library and load **SmolVLA**, the actual 450M parameter Vision-Language-Action model from the Hugging Face LeRobot team.

This step is optional. LeRobot is a young library and its install sometimes conflicts with whatever PyTorch version Colab shipped today. If it works, you will see SmolVLA's parameter count and its top level modules. If it does not, you will see a clean message telling you so, and the main point of this module is already done.

In [ ]:
!pip install -q 'lerobot[smolvla]' 2>&1 | tail -n 5

In [ ]:
try:
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    policy = SmolVLAPolicy.from_pretrained('lerobot/smolvla_base')
    n_params = sum(p.numel() for p in policy.parameters())
    print('Loaded lerobot/smolvla_base')
    print(f'Total parameters: {n_params/1e6:.1f}M')
    print()
    print('Top level modules inside SmolVLA:')
    for name, _ in policy.named_children():
        print(' -', name)
except Exception as e:
    print('SmolVLA did not load cleanly in this environment.')
    print('That is OK. You already saw the vision language half work above,')
    print('and we will come back to the full VLA in a later module.')
    print()
    print('Error was:')
    print(repr(e))

## What you just did

1. You loaded a 500M parameter multimodal model and asked it a question about an image. It answered in fluent English.
2. You saw, with your own eyes, that the vision language brain of a VLA is not magic. It is a file you can download and run in a browser.
3. You optionally loaded the full SmolVLA and inspected its architecture.

You are now standing at the finish line of the course. In the next 19 modules, we will walk back to the start and build up every single piece of what you just ran, from NumPy and gradient descent, to the Transformer, to modern robotics math, all the way back to fine tuning SmolVLA on a new robot task.

See you in Module 1.